# ASTR 457: The Midterm

Posted Thu Oct 8, due Wed Oct 14 by Noon (fork → PR, as always). Graded like a lab.

Three parts. Each is designed to test something every lab so far has come back to - which error bar is the right one, whether the model needs the extra parameter, and whether the check you ran could have failed. You've now got the tools to do all of this, so let's do it on numbers you haven't seen: 250 real Type Ia supernovae, and someone else's analysis of real white dwarfs.

Part 1 is 30 pts (6 each), Part 2 is 45 pts, and Part 3 is 25 pts. You'll need 2b and 2c right to get 2d right.

Remember, the rules are the syllabus rules - open book, you can use AI tools (document them in 3b), you can work together and discuss problems and solutions, but you have to write up your own solution and run it on **your own data** in `data/<your netid>/` (nobody else's numbers will match yours).

Put your notebook in `submissions/<your netid>/midterm.ipynb`. The path below (`DATA`) reaches your files from there. Make sure it runs top to bottom from a fresh kernel ("Restart & Run All") - that is how it will be graded, and how it will be run if you defend it.

Throughout, make sure every number you quote has a unit, an error bar and a sensible number of digits (two significant figures on the error bar, and round the value to match).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

NETID = 'benh7'   # <-- set this
DATA = f'../../labs/midterm/data/{NETID}/'   # from submissions/<your netid>/

def load(name):
    """Read one of your CSV files as a pandas table; d['column'].to_numpy() gives a numpy array. Names stay strings."""
    return pd.read_csv(DATA + name + '.csv', dtype={'name': str})

## Part 1: Error bars (30 pts, 6 each)

Five short scenarios. Each one is a number someone handed you - say whether it's right, compute the right one, and say in a sentence or two why. Show the code. Each of these should be about 20 minutes of work.

### 1a - The extinction coefficient (`p1a_extinction.csv`)

An observer measured a photometric zero point at 15 different airmasses through one night. Extinction in magnitudes grows in proportion to the airmass $X$, so the zero point should follow a line, $m = k\,X + m_0$, where $k$ is the extinction coefficient in mag per airmass. Columns: `airmass`, `zeropoint_mag`, `sigma_quoted_mag` (the pipeline's error on each point).

1. Fit the line by weighted least squares with the quoted errors. Report $k$ and its error bar computed from the quoted errors alone.
2. Now run `np.polyfit(airmass, zeropoint_mag, 1, w=1/sigma_quoted_mag, cov=True)` and report its error bar on $k$. Explain exactly what `polyfit` did to get a different number (read its documentation for `cov`).
3. Compute $\chi^2$ and the number of degrees of freedom. How probable is a $\chi^2$ this large or this small if the quoted errors were right? (Compute it from the $\chi^2$ distribution.)
4. Report the error bar on $k$ you'd quote, and justify it. What does your answer to (3) tell you about the pipeline's error bars?

In [ ]:
# 1a: your work here
df = load('p1a_extinction')
df

In [ ]:
X = np.column_stack([np.ones_like(df['airmass']), df['airmass']])
E = np.eye(len(df['sigma_quoted_mag'])) * np.array(df['sigma_quoted_mag']**2)

theta_hat = np.linalg.inv(X.T @ np.linalg.inv(E) @ X) @ (X.T @ np.linalg.inv(E) @ np.array(df['zeropoint_mag']))

plt.errorbar(df['airmass'], df['zeropoint_mag'], yerr = df['sigma_quoted_mag'], fmt = '.', ecolor = 'black', capsize = 2)
plt.title('')
plt.xlabel('Airmass')
plt.ylabel('Zeropoint Mag')
plt.grid(linestyle = '--', linewidth = 0.5)

y_w_intercept = theta_hat[0]
w_slope = theta_hat[1]

x_w_slope = np.linspace(1, 2.25, 1000)
y_w_slope = w_slope * x_w_slope + y_w_intercept

plt.plot(x_w_slope, y_w_slope, color = 'grey', linestyle = '--', label = f'Weighted OLS fit: y = {w_slope:.3f}x + {y_w_intercept:.3f}')
plt.legend()

In [ ]:
w_cov = np.linalg.inv(X.T @ np.linalg.inv(E) @ X)
w_intercept_err = np.sqrt(w_cov[0, 0])
w_slope_err = np.sqrt(w_cov[1, 1])

print(f'K: {w_slope:.2f} \u00B1 {w_intercept_err:.2f}')

In [ ]:
fit, np_cov = np.polyfit(df['airmass'], df['zeropoint_mag'], deg = 1, w = 1 / (df['sigma_quoted_mag']), cov = True)
uncert = np.sqrt(np_cov[0][0])
print(f'np.polyfit K: {fit[0]:.2f} {uncert:.2f}')

Since cov = True, np.polyfit will scale the covariance by chi2/dof. Due to this scaling relation, the improper weights are handed to the program and it uses its own system of weighting the errors. This alternate system is incorrect however as there is not a uniform sigma for the errors but rather differently weighted errors per point.

In [ ]:
DOF = 2
Residuals = df['zeropoint_mag'] - (w_slope * df['airmass'] + y_w_intercept)
Chi2 = np.sum((Residuals/df['sigma_quoted_mag'])**2)
print(f'Chi2: {Chi2}')

In [ ]:
data = np.zeros(5000)

for i in range(5000):
    rand_sig = np.random.choice(np.array(df['sigma_quoted_mag']), size = 15, replace = True)
    rand_Chi2 = np.sum((Residuals / rand_sig)**2)
    data[i] = rand_Chi2

In [ ]:
plt.hist(data, bins = 30)
plt.title('Distribution of Chi2 from Variations in Error')
plt.xlabel('Chi2')
plt.ylabel('Count')
plt.vlines(Chi2, ymin = 0, ymax = 400, linestyle = '--', color = 'black', label = f'Chi2: {Chi2:.2f}')

lo, hi = np.percentile(data, [16, 84])
plt.axvspan(lo, hi, color = 'green', alpha = .3, label = f'16th percentile {lo:.2f}\n84th percentile {hi:.2f}')
plt.ylim([0, 400])
plt.grid(linestyle = '--', linewidth = 0.5)
plt.legend()

In [ ]:
import scipy
chance = scipy.stats.percentileofscore(data, Chi2)
print(f'Chi2 has Percentile of: {chance:.2f}')
print(f'This is outside a 1 sigma uncertainty.')

Since this is outside the 1 sigma range, there is evidence to suggest that not all the errors in the dataset are explained by the given varience.

**ANSWER:**

*K = .16 +- .02* 

It should be noted: As determined in part three, these reported values are biased as they do not account for the full width of errors.

### 1b - The centroid (`p1b_centroid.csv`)

A classmate measured the position of a star from `n_photons` photons, each landing at a position drawn from a Gaussian of width `psf_sigma_arcsec` around the true position. They report an error on the position of `quoted_sigma_x0_arcsec`, from a clever weighting scheme they found online.

1. Compute the Cramér-Rao bound on the error of any unbiased estimator of the position (Lab 02).
2. Is the quoted error possible? If not, say what the classmate's number must mean instead, and name two concrete mistakes that would produce an error bar this small.

In [ ]:
# 1b: your work here
df2 = load('p1b_centroid')
df2

In [ ]:
CRB = df2['psf_sigma_arcsec'][0] / np.sqrt(df2['n_photons'][0])
CRB

**ANSWER:**

*Cramer-Rao Bound: .04*

*quoted_sigma_x0_arcsec < Cramer-Rao bound*

This quoted error is not possible as it is smaller than the Cramer-Rao bound. If the estimator was unbiased, the smallest error it could possibly have is about .041. Some potential reasons this would produce an error too small are:

1) Underestimated errors: The student may be overly confident in their method and be placing overconfident errorbars on their data. If the errors were smaller then they are supposed to be, then this could make the stated errors smaller than the Cramer-Rao bound. If done however, this would mean that the errorbars are no longer catching 1 sigma's worth of values which would result in faulty data ranges.
2) Biased estimator: The student may have some underlying assumptions in their model that are incorrect about their data (perhaps they assume the data is Gaussian when it takes on a different shape). This bias may shrink the reported errors due to the different mehodology. This still results in errorbars that do not fully account for the error in a 1 sigma range indicating that the bound does not fully capture the true center.

### 1c - The transit depth's error bar (`p1c_profile.csv`)

A transit-depth fit produced this profile: columns `depth_ppt` (the depth, in parts per thousand) and `minus2lnL` ($-2\ln L$ at that depth, other parameters fixed at their best values). A classmate took the second derivative of `minus2lnL` at its minimum and reported $\sigma = 1/\sqrt{d^2(-2\ln L)/d\,\textrm{depth}^2}$.

1. Find the best depth and the correct $1\sigma$ error from the curvature. Is the classmate's error bar right? If not, by what factor is it off, and why?
2. Read the $\Delta(-2\ln L) = 1$ interval directly off the profile. Is it symmetric? Compare it with your curvature answer.
3. If the classmate quotes $\pm$ their $\sigma$ on many independent datasets, what fraction of those intervals contain the true depth? Predict it, then check it with a simulation (a few thousand fake best-fit depths drawn around a truth you choose, using your correct $\sigma$).

In [ ]:
# 1c: your work here
df3 = load('p1c_profile')
df3

In [ ]:
plt.scatter(df3['depth_ppt'], df3['minus2lnL'])
plt.title('Depth VS -2lnL')
plt.xlabel('Depth [ppt]')
plt.ylabel('-2lnL')
plt.grid(linestyle = '--', linewidth = 0.5)

In [ ]:
depth_ppt = df3['depth_ppt'].to_numpy()
minus2lnL = df3['minus2lnL'].to_numpy()

idx = np.argmin(minus2lnL)

minus2lnL_min = minus2lnL[idx]
depth_ppt_min = depth_ppt[idx]

In [ ]:
h = depth_ppt[idx+1] - depth_ppt[idx]

curvature = (minus2lnL[idx+1] - 2 * minus2lnL[idx] + minus2lnL[idx-1] ) / h**2

sigma_Hessian = np.sqrt(2/curvature)

sigma_Hessian

The estimated uncertainty is about .707 times too small than it should be. This comes from the fact that the there is a factor of 2 that gets carried through the calculation. The formula would've been correct if they were given the -lnL. 

In [ ]:
plt.scatter(df3['depth_ppt'], df3['minus2lnL'])
plt.title('Depth VS -2lnL')
plt.xlabel('Depth [ppt]')
plt.ylabel('-2lnL')
plt.hlines(1 + minus2lnL_min, xmin = 2, xmax = 4.5, linestyle = '--',color = 'black')
plt.ylim([108.25, 111])
plt.xlim([2.75, 3.75])

plt.scatter(depth_ppt[idx + 10], minus2lnL[idx + 10], color = 'red')
plt.scatter(depth_ppt[idx - 10], minus2lnL[idx - 10], color = 'red')

plt.grid(linestyle = '--', linewidth = 0.5)
plt.legend(['data', '1 above -2lnL minimum', 'Intersected Points'], loc = 'lower right')

In [ ]:
print(depth_ppt[idx + 10] - depth_ppt[idx])

In [ ]:
print(depth_ppt[idx] - depth_ppt[idx - 10])

These values are equal on either side of the curve indicating this is a symmetric curve. These values are also essentially the calculated uncertainty from the Hessian (same exact value when rounded to the third decimal place). This means the curveture is in line with the estimation of the error. 

Prediction: The correct values should be off by about a factor of .707. This means that 68% * .707 = 52% of the values should be correct.

In [ ]:
true_val = depth_ppt[idx]
simulation = np.random.normal(true_val, sigma_Hessian, 5000)

under_covered = np.where( (true_val - np.sqrt(1/curvature) < simulation) & (simulation < true_val + np.sqrt(1/curvature) ) )

small_errs = len(simulation[under_covered])/5000

true_covered = np.where( (true_val - sigma_Hessian < simulation) & (simulation < true_val + sigma_Hessian ) )

true_errs = len(simulation[true_covered])/5000

plt.bar(['True Errors', 'Small Errors'], [true_errs, small_errs])
plt.title('Comparison of Errors')
plt.ylabel('% of True Values Covered')
plt.grid(linestyle = '--', linewidth = 0.5)

**ANSWER:**

*The true value of the small error bars lies in the stated range ~52% of the time NOT 68% of the time.*

*True Errors: 0.32 m ppt*

### 1d - The companion mass (`p1d_mass_samples.csv`)

These are 8000 posterior samples of the mass of an unseen companion, `M2_Msun`. Three classmates report its uncertainty as (i) the variance of the samples, (ii) the 84th percentile minus the 16th, and (iii) the standard deviation divided by $\sqrt{8000}$.

1. Report the mass as a median with its 16th-84th percentile error bars, in solar masses.
2. For each of (i)-(iii): what quantity is it actually, what are its units, and by roughly what factor is it wrong as a $1\sigma$ error bar?
3. Is a symmetric $\pm$ error bar appropriate for this posterior? If not, show why.

In [ ]:
# 1d: your work here
df4 = load('p1d_mass_samples')
df4

In [ ]:
plt.hist(df4['M2_Msun'], bins = 30)
plt.title('Distribution of Posterior Mass samples')
plt.xlabel('Mass [Solar Masses]')
plt.ylabel('Count')

M_lo, M_m, M_hi = np.percentile(np.array(df4['M2_Msun']), [16, 50, 84])

plt.vlines(M_m, ymin = 0, ymax = 1250, linestyle = '--', color = 'black', label = f'Median: {M_m:.2f}')
plt.axvspan(M_lo, M_hi, color = 'Green', alpha = .3, label = f'16th percentile: {M_lo:.2f}\n84th percentile: {M_hi:.2f}')
plt.ylim([0, 1200])
plt.legend()
plt.grid(linestyle = '--', linewidth = 0.5)

In [ ]:
Upper_bound = M_hi - M_m
Lower_bound = M_m - M_lo

print(f'Mass: {M_m:.1f} Solar Masses + {Upper_bound:.1f} Solar Masses or - {Lower_bound:.1f} Solar Masses')

In [ ]:
#i
variance = np.var(df4['M2_Msun'])
width_true = Lower_bound + Upper_bound
width_var = 2 * variance

var_estimation = width_var / width_true

print(f'Mass using variance: {M_m:.1f} \u00B1 {variance:.1f} Solar Masses^2')
print('This is the variance. It is the (standard deviation)^2 for a symmetric gaussian.')
print(f'The mass range using variance is about {var_estimation:.1f} times larger than it should be.')

In [ ]:
#ii
diff = M_hi - M_lo
diff_shift = (2 * diff) / width_true  

print(f'Mass using the 84th percentile - 16th percentile: {M_m:.1f} \u00B1 {diff:.1f} Solar Masses')
print('This is the width of the middle 68%.')
print(f'The mass range using the 84th percentile - 16th percentile is about {diff_shift} times larger than it should be.')

In [ ]:
#iii
std = df4['M2_Msun'].std()
standard_error = std / np.sqrt(8000)
width_se = 2 * standard_error 

se_estimation = width_se / width_true

print(f'Mass using standard error: {M_m:.2f} \u00B1 {standard_error:.2f} Solar Masses')
print('This is the standard error. This gives the standard deviation of the mean.')
print(f'The mass range using the standard error is about {se_estimation:.2f} times the size it should be.')

Symmetric error bars are not approprate for this graph. This is as the graph has a tail on the right side of the graph. This drags the range of values out about the right side. If the error bars were symmetric, it would not include the true 84th percentile.  

In [ ]:
plt.hist(df4['M2_Msun'], bins = 30)
plt.title('Distribution of Posterior Mass samples')
plt.xlabel('Mass [Solar Masses]')
plt.ylabel('Count')

plt.vlines(M_m, ymin = 0, ymax = 1250, linestyle = '--', color = 'black', label = f'Median: {M_m:.2f}')
plt.axvspan(M_lo, M_hi, color = 'Green', alpha = .3, label = f'16th percentile: {M_lo:.2f}\n84th percentile: {M_hi:.2f}')
plt.axvspan(M_m - diff/2, M_m + diff/2, color = 'red', alpha = .3, label = f'16th percentile: {(M_m - diff/2):.2f}\n84th percentile: {(M_m + diff/2):.2f}')
plt.ylim([0, 1200])
plt.legend()
plt.grid(linestyle = '--', linewidth = 0.5)

**ANSWER:**

*your answer goes here*

### 1e - The slope at fixed intercept (`p1e_slope_intercept_samples.csv`)

Posterior samples of a slope and an intercept from a line fit. A classmate fixed the intercept at its best value, refit only the slope, and quoted the slope's error from that 1-D fit.

1. Compute the slope's error bar the classmate got (the spread of the slope *at fixed intercept*), and the marginal one (the spread of the slope with the intercept marginalized over).
2. Compute their ratio, and the correlation coefficient $\rho$ of the samples. How does the ratio depend on $\rho$? Plot the samples and draw both error bars on the plot.
3. When is the classmate's number the right one to quote? One sentence.

In [ ]:
# 1e: your work here
df5 = load('p1e_slope_intercept_samples')
df5

In [ ]:
plt.scatter(df5['intercept'], df5['slope'], alpha = .2, label = 'data')
plt.xlabel('Intercept')
plt.ylabel('Slope')
plt.title('Intercept VS Slope')
plt.grid(linestyle = '--', linewidth = 0.5)
plt.legend()

In [ ]:
#What pandas yields for r
df5.corr()

In [ ]:
sigma_marginal = df5['slope'].std()
sigma_marginal

In [ ]:
rho = np.array(df5.corr())[0][1]

sigma_fixed = sigma_marginal * np.sqrt(1 - rho**2)
sigma_fixed

In [ ]:
ratio = sigma_fixed / sigma_marginal
ratio

In [ ]:
np.sqrt(1 - rho**2)

ratio dependence on rho: As rho increases, the error bars become larger underestimate of true value.

In [ ]:
intercept_med = df5['intercept'].median()
slope_med = df5['slope'].median()

plt.figure(figsize=(16,6))
plt.subplot(1, 2, 1)

plt.scatter(df5['intercept'], df5['slope'], alpha=0.2)
plt.errorbar(intercept_med, slope_med, yerr = sigma_marginal, color = 'red', fmt = 'o', capsize = 5, label = 'Marginalized slope uncertainty')
plt.errorbar(intercept_med, slope_med, yerr = sigma_fixed, color = 'black', fmt = 'o', capsize = 5, label = 'Fixed-intercept uncertainty')

plt.axvline(intercept_med, color = 'black', linestyle = '--', alpha = .2, label = 'Projection Axis')

plt.xlabel('Intercept')
plt.ylabel('Slope')
plt.title('Slope-Intercept Posterior Samples')
plt.legend()
plt.grid(linestyle = '--', linewidth = 0.5)



plt.subplot(1, 2, 2)
plt.hist(np.array(df5['slope']), bins = 30)
plt.title('Histogram of Posterior Sample Slopes')
plt.xlabel('Slopes')
plt.ylabel('Count')

plt.axvline(slope_med, color = 'black', linestyle = '--', label = 'slope median')

plt.axvspan(slope_med + sigma_marginal, slope_med - sigma_marginal, color = 'green', alpha = .3, label = 'Sigma Marginal')
plt.axvspan(slope_med + sigma_fixed, slope_med - sigma_fixed, color = 'red', alpha = .3, label = 'Sigma Fixed')
plt.legend()
plt.grid(linestyle = '--', linewidth = 0.5)

print(f'The 16th percentile is: {np.percentile(df5['slope'], 16)}')
print(f'The 84th percentile is: {np.percentile(df5['slope'], 84)}')

The student's value is the right to quote when values are uncorrelated.

**ANSWER:**

*True Errors about the median:*

## Part 2: 250 supernovae and a step (45 pts)

Type Ia supernovae are standardizable candles: their peak brightness, corrected for how fast they fade and for their color, measures distance well enough to have discovered cosmic acceleration. Your file `p2_pantheon_sne.csv` holds 250 real SNe Ia from the Pantheon+ compilation (Scolnic et al. 2022, ApJ 938, 113; [Brout et al. 2022](https://arxiv.org/abs/2202.04077), ApJ 938, 110), all at redshift $0.023 < z < 0.15$. For each supernova, a light-curve fit (SALT2) gives:

- `mB`: the peak apparent magnitude in the B band, and `mB_err`;
- `x1`: the light-curve *stretch* (broader light curves are brighter), and `x1_err`;
- `c`: the *color* (redder supernovae are fainter, from dust and intrinsically), and `c_err`;
- `cov_mB_x1`, `cov_mB_c`, `cov_x1_c`: the covariances between those three, from the same light-curve fit;
- `z`: the redshift; `host_logmass`, `host_logmass_err`: $\log_{10}$ of the host galaxy's stellar mass in solar masses, and its error.

The standardized brightness is the **Tripp relation**,

$$m_B = M + \mu(z) - \alpha\,x_1 + \beta\,c + \gamma\,\Theta(\log M_\star - 10),$$

where $M$ is the absolute magnitude of a "standard" SN Ia, $\mu(z)$ the distance modulus, $\Theta$ is 1 for hosts more massive than $10^{10}\,M_\odot$ and 0 otherwise, and $\gamma$ is the **host-mass step**. Supernovae in massive hosts appear to be brighter after standardization (Kelly et al. 2010; Sullivan et al. 2010); whether the step is real, and why, matters for supernova dark-energy measurements. In this form, a step that makes supernovae in massive hosts brighter has $\gamma < 0$; the published value is about $\gamma \approx -0.06$ mag. So the question for your sample is whether you need $\gamma$ at all.

There are two more things you'll need. At these redshifts the galaxies' own motions (peculiar velocities, about 250 km/s) shift $z$, and so $\mu$; add $\sigma_{\mu,\rm pec} = \frac{5}{\ln 10}\frac{250\ {\rm km/s}}{c\,z}$ to each supernova's variance. And no supernova is a perfect standard candle: add an intrinsic scatter $\sigma_{\rm int}$, a free parameter. Use a flat $\Lambda$CDM distance modulus with $H_0 = 70$ km/s/Mpc and $\Omega_m = 0.3$ (`astropy.cosmology.FlatLambdaCDM(H0=70, Om0=0.3).distmod(z)`); $M$ absorbs the choice of $H_0$.

We planted between **two and four contaminants** in your file - objects whose `mB` we made fainter by more than a magnitude, so they are not standard SNe Ia any more (think of a mis-classified peculiar supernova). Their names are not in the file. Real data have a few real outliers too, so not everything far from the line is planted.

### 2a - Look at the data (5 pts)

1. Compute $\mu(z)$ and plot the **Hubble residuals** $m_B - \mu(z)$ against $x_1$, against $c$, and against `host_logmass`.
2. Which way do the $x_1$ and $c$ trends go? Is that the sign the Tripp relation expects? Two or three sentences.
3. The raw trend with host mass may not have the sign of $\gamma$. Why not? (Plot $x_1$ against host mass.)
4. Can you see any of the contaminants?
5. What units do $\alpha$ and $\beta$ have?

In [ ]:
# 2a: your work here
df6 = load('p2_pantheon_sne')
df6

In [ ]:
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u

cosmo = FlatLambdaCDM(H0 = 70, Om0 = 0.3)

mu = cosmo.distmod(df6['z']).value
hubble_residual = np.array(df6['mB']) - mu
hubble_residual

In [ ]:
plt.figure(figsize = (20, 6))
plt.subplot(1, 3, 1)
plt.scatter(df6['x1'], hubble_residual)
plt.title('x1 VS Hubble Residual')
plt.xlabel('x1')
plt.ylabel('Hubble Residaul')
plt.grid(linestyle = '--', linewidth = 0.5)

plt.subplot(1, 3, 2)
plt.scatter(df6['c'], hubble_residual)
plt.title('c VS Hubble Residual')
plt.xlabel('c')
plt.ylabel('Hubble Residaul')
plt.grid(linestyle = '--', linewidth = 0.5)

plt.subplot(1, 3, 3)
plt.scatter(df6['host_logmass'], hubble_residual)
plt.title('host_logmass VS Hubble Residual')
plt.xlabel('host_logmass')
plt.ylabel('Hubble Residaul')
plt.grid(linestyle = '--', linewidth = 0.5)

This is what is expected for x1 and c. As the supernova light gets streched out it becomes brighter. Similarly as the color becomes redder the light becomes dimmer.  

In [ ]:
plt.scatter(df6['x1'], df6['host_logmass'])
plt.title('x1 VS host_logmass')
plt.xlabel('x1')
plt.ylabel('host_logmass')
plt.grid(linestyle = '--', linewidth = 0.5)

The raw data file for gamma may not follow the stated trend. This is as x1 and host_logmass are correlated; the correlation confounds the trend as changes in x1 will also affect the host_logmass of the system. This causes additional structure in the raw trend as there are multiple competing factors that influence the final result. 

There are some points points in the c VS hubble residual graph that appear to be significantly fainter than the rest of the points by about a magnitude; I can't be fully certain that this is the case since there hasn't been a scientific comparison made for them.

Both alpha and beta have units of magnitudes. 

**ANSWER:**

*your answer goes here*

### 2b - The likelihood (10 pts)

The parameters are $\theta = (M, \alpha, \beta, \gamma, \sigma_{\rm int})$. Each supernova's variance is

$$\sigma_i^2 = \mathbf{v}^T C_i\,\mathbf{v} + \sigma_{\mu,{\rm pec},i}^2 + \sigma_{\rm int}^2, \qquad \mathbf{v} = (1, \alpha, -\beta),$$

where $C_i$ is that supernova's $3\times3$ covariance of $(m_B, x_1, c)$.

1. Write out $\mathbf{v}^T C_i\,\mathbf{v}$ term by term, and explain in one sentence why the off-diagonal terms are there.
2. Write the log-likelihood. Remember the $\ln\sigma_i^2$ term - the variance depends on the parameters.
3. State a prior for every parameter, and justify each in one line.
4. Decide whether to sample $\ln\sigma_{\rm int}$ or $\sigma_{\rm int}$, and say why.
5. Decide what to do about the contaminants. You may model them or remove them, but say how you found them and what a cut would cost you.

In [ ]:
# 2b: your work here

def log_liklihood(M, a, B, gamma, sigmaInt, delta, sigmaCont, f):

    VTCV = (df6['mB_err']**2) + (a**2 * df6['x1_err']**2) + (B**2 * df6['c_err']**2) + (2 * a * df6['cov_mB_x1']) - (2 * B * df6['cov_mB_c']) - (2 * a * B * df6['cov_x1_c'])
    
    c_light = 299792.458 #km/s
    sigmaMuPec = (5 / np.log(10)) * (250) / (c_light * df6['z'])

    sigmaSqr = VTCV + sigmaMuPec**2 + sigmaInt**2

    theta = (df6['host_logmass'] > 10).astype(int)

    r = df6['mB'] - (M + mu - a * df6['x1'] + B * df6['c'] + gamma * theta)

    L_SN = (1 / np.sqrt(2 * np.pi * sigmaSqr)) * np.exp(-.5 * r**2 / sigmaSqr) 
           
    L_cont = ( (1 / np.sqrt(2 * np.pi *sigmaCont**2) * np.exp(-0.5 * (r - delta)**2 / sigmaCont**2)) ) 

    Ltot = (1 - f) * L_SN + f * L_cont
    
    return np.sum(np.log(Ltot))

These off-diagonal terms are here as the SALT2 model propogates errors through its modeling process. As the errors are correlated, these values need to be included to ensure that the proper sigma is calculated.

Priors:

1) M: -21 < M < -15 ; Bounds are typical magnitudes of superluminous and subluminous supernovae. Type Ia should be between this (otherwise they'd be classified differently)
2) a: 0 < a ; Stretch is a positive value; can't be negativly stretched.
3) B: 0 < B ; redder supernovae become brighter. Can't become negatively red. 
4) gamma: -.2 < gamma < .2 ; published value is about -.06. range of .2s prevents gamma from becoming orders of magnitude larger than published results 
5) ln(sigmaInt): ln(sigmaInt) < ln(1); Intrinsic scattering is positive value. ln(1) is order of magnitude larger than largest uncertainty; larger scattering would be unrealistic
6) Delta: .75 < delta < 10; containments are spread by more than 1 magnitude. This provides a slight underbound and order of magnitude larger to catch faulty data. 
7) ln(SigmaCont): sigmaCont < ln(5); spread of contaniments is positive value 5 is order of magnitude larger than largest uncertainty; broader than sigmaInt as it has larger spread due to mislabeling
8) f: 0 < f < .1; fraction of containments should be small ~2-5 per CSV; gives order of magnitude larger estimate for what fraction should reasonably be. 

In [ ]:
def log_prior(theta):
    M, a, B, gamma, log_int, delta, log_cont, log_f = theta

    if (M < -21) | (-15 < M):
        return -np.inf

    if 0 >= a:
        return -np.inf

    if 0 >= B:
        return -np.inf

    if ((gamma < -.2) | (.2 < gamma)):
        return -np.inf

    if (log_int < np.log(1e-4)) | (0 < log_int):
        return -np.inf

    if (delta < 0) | (1.5 < delta):
        return -np.inf

    if (log_cont < np.log(1e-4)) | (np.log(5) < log_cont):
        return -np.inf

    if (log_f < np.log(1/250)) | (np.log(.1) < log_f):
        return -np.inf

    return 0.0

I plan to sample using ln(sigmaInt). This builds the prior that sigmaInt is a positive value into the scheme without placing a hard wall at 0. This allows for a bit softer of an exploration for the sampler.

I am modeling the contaniments in my liklihood. I added an extra liklihood term for some fraction of stars that are contamined by some random magnitude larger than 1 and are spread about as a gaussian. When the MCMC runs I will get sample values for this term to find true fraction of contaminated stars and see how they are spread about my data. This prevents data loss and keeps the information within the system.

**ANSWER:**

*your answer goes here*

### 2c - Sampling and convergence (12 pts)

1. Find the maximum a posteriori point. Then run **four independent chains** from starting points spread wider than the posterior (you choose how, and say how you chose). Any sampler is fine - your own Metropolis, `emcee` (one ensemble per chain), or `PyMC`.
2. Report $\hat R$ and the effective sample size for every parameter (`arviz.rhat`, `arviz.ess`, or `arviz.summary`), and say what values would have made you run longer.
3. Make a corner plot, and quote $\alpha$, $\beta$, $\gamma$ and $\sigma_{\rm int}$ as medians with 16th-84th percentile error bars.
4. Which parameters are correlated strongly? Comment on why. Then fix $M$ at its MAP value, refit $\gamma$, and compare that $\sigma_\gamma$ with the marginal one (cf. 1e).

`ArviZ` 1.x (install it with `conda install -c conda-forge pymc arviz`, as in `INSTALL.txt`) takes your chains as a dictionary of arrays shaped (chain, draw): `idata = az.from_dict({'posterior': {'alpha': a, 'beta': b}})`, then `az.summary(idata)`.

In [ ]:
# 2c: your work here
def neg_log_posterior(theta):

    lp = log_prior(theta)

    if not np.isfinite(lp):
        return np.inf

    M, a, B, gamma, log_int, delta, log_cont, log_f = theta

    sigmaInt = np.exp(log_int)
    sigmaCont = np.exp(log_cont)
    f = np.exp(log_f)

    return -(lp + log_liklihood(M, a, B, gamma, sigmaInt, delta, sigmaCont, f))

In [ ]:
from scipy.optimize import minimize

guess = [-19.3, 0.15, 3.0, -0.05, np.log(0.1), 1.1, np.log(1.0), np.log(.02)]
result = minimize(neg_log_posterior, guess)
print(result.x)

In [ ]:
import emcee
pos = result.x + 1e-4 * np.random.randn(32, 8)

nwalkers, ndim = pos.shape

def log_posterior(val):
    return -1 * neg_log_posterior(val)

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior)

sampler.run_mcmc(pos, 20_000, progress = False)
print('')

In [ ]:
fig, axes = plt.subplots(8, figsize=(10, 7), sharex=True)
samples = sampler.get_chain()
labels = ["M", "a", "B", "gamma", "log_int", "delta", "log_cont", "log_f"]
for i in range(ndim):
    ax = axes[i]
    ax.plot(samples[:, :, i], "k", alpha=0.3)
    ax.set_xlim(0, len(samples))
    ax.set_ylabel(labels[i])
    ax.yaxis.set_label_coords(-0.1, 0.5)

axes[-1].set_xlabel("step number")

print(f"Mean acceptance fraction: {sampler.acceptance_fraction.mean():.2f} (0.2-0.5 is healthy)")

In [ ]:
pos2 = result.x + 1e-3 * np.random.randn(32, 8)

nwalkers2, ndim2 = pos.shape

sampler2 = emcee.EnsembleSampler(nwalkers2, ndim2, log_posterior)

sampler2.run_mcmc(pos2, 20_000, progress = False)
print('')

In [ ]:
fig, axes = plt.subplots(8, figsize=(10, 7), sharex=True)
samples2 = sampler2.get_chain()
labels = ["M", "a", "B", "gamma", "log_int", "delta", "log_cont", "log_f"]
for i in range(ndim2):
    ax = axes[i]
    ax.plot(samples2[:, :, i], "k", alpha=0.3)
    ax.set_xlim(0, len(samples2))
    ax.set_ylabel(labels[i])
    ax.yaxis.set_label_coords(-0.1, 0.5)

axes[-1].set_xlabel("step number")

print(f"Mean acceptance fraction: {sampler.acceptance_fraction.mean():.2f} (0.2-0.5 is healthy)")

In [ ]:
pos3 = result.x + 1e-2 * np.random.randn(32, 8)

nwalkers3, ndim3 = pos.shape

sampler3 = emcee.EnsembleSampler(nwalkers3, ndim3, log_posterior)

sampler3.run_mcmc(pos3, 20_000, progress = False)
print('')

In [ ]:
fig, axes = plt.subplots(8, figsize=(10, 7), sharex=True)
samples3 = sampler3.get_chain()
labels = ["M", "a", "B", "gamma", "log_int", "delta", "log_cont", "log_f"]
for i in range(ndim3):
    ax = axes[i]
    ax.plot(samples3[:, :, i], "k", alpha=0.3)
    ax.set_xlim(0, len(samples3))
    ax.set_ylabel(labels[i])
    ax.yaxis.set_label_coords(-0.1, 0.5)

axes[-1].set_xlabel("step number")

print(f"Mean acceptance fraction: {sampler.acceptance_fraction.mean():.2f} (0.2-0.5 is healthy)")

In [ ]:
pos4 = result.x + 3e-2 * np.random.randn(32, 8)

nwalkers4, ndim4 = pos.shape

sampler4 = emcee.EnsembleSampler(nwalkers4, ndim4, log_posterior)

sampler4.run_mcmc(pos4, 20_000, progress = False)
print('')

In [ ]:
fig, axes = plt.subplots(8, figsize=(10, 7), sharex=True)
samples4 = sampler4.get_chain()
labels = ["M", "a", "B", "gamma", "log_int", "delta", "log_cont", "log_f"]
for i in range(ndim4):
    ax = axes[i]
    ax.plot(samples4[:, :, i], "k", alpha=0.3)
    ax.set_xlim(0, len(samples4))
    ax.set_ylabel(labels[i])
    ax.yaxis.set_label_coords(-0.1, 0.5)

axes[-1].set_xlabel("step number")

print(f"Mean acceptance fraction: {sampler.acceptance_fraction.mean():.2f} (0.2-0.5 is healthy)")

In [ ]:
!pip install pymc arviz

In [ ]:
import arviz as az

idata1 = az.from_dict({'posterior' : {
        "M": samples[:, :, 0].T,
        "a": samples[:, :, 1].T,
        "B": samples[:, :, 2].T,
        "gamma": samples[:, :, 3].T,
        "log_int": samples[:, :, 4].T,
        "delta": samples[:, :, 5].T,
        "log_cont": samples[:, :, 6].T,
        "log_f": samples[:, :, 7].T,
    }}
)

idata2 = az.from_dict({'posterior' : {
        "M": samples2[:, :, 0].T,
        "a": samples2[:, :, 1].T,
        "B": samples2[:, :, 2].T,
        "gamma": samples2[:, :, 3].T,
        "log_int": samples2[:, :, 4].T,
        "delta": samples2[:, :, 5].T,
        "log_cont": samples2[:, :, 6].T,
        "log_f": samples2[:, :, 7].T,
    }}
)

idata3 = az.from_dict({'posterior' : {
        "M": samples3[:, :, 0].T,
        "a": samples3[:, :, 1].T,
        "B": samples3[:, :, 2].T,
        "gamma": samples3[:, :, 3].T,
        "log_int": samples3[:, :, 4].T,
        "delta": samples3[:, :, 5].T,
        "log_cont": samples3[:, :, 6].T,
        "log_f": samples3[:, :, 7].T,
    }}
)

idata4 = az.from_dict({'posterior' : {
        "M": samples4[:, :, 0].T,
        "a": samples4[:, :, 1].T,
        "B": samples4[:, :, 2].T,
        "gamma": samples4[:, :, 3].T,
        "log_int": samples4[:, :, 4].T,
        "delta": samples4[:, :, 5].T,
        "log_cont": samples4[:, :, 6].T,
        "log_f": samples4[:, :, 7].T,
    }}
)

In [ ]:
az.summary(idata)

In [ ]:
az.summary(idata2)

In [ ]:
az.summary(idata3)

In [ ]:
az.summary(idata4)

In [ ]:
tau = sampler2.get_autocorr_time()
tau_max = np.max(tau)
print(tau_max)

In [ ]:
tau_max = 280

In [ ]:
import corner
flat_samples = sampler2.get_chain(discard = tau_max * 3, thin = 140, flat = True)

MCMCM = flat_samples.T[0]
MCMCa = flat_samples.T[1]
MCMCB = flat_samples.T[2]
MCMCgamma = flat_samples.T[3]
MCMCTlog_int = flat_samples.T[4]
MCMCdelta = flat_samples.T[5]
MCMClog_cont = flat_samples.T[6]
MCMClog_f = flat_samples.T[7]

MCMCint = np.exp(MCMCTlog_int)
MCMCcont = np.exp(MCMClog_cont)
MCMCf = np.exp(MCMClog_f)

MCMC = np.array([MCMCM, MCMCa, MCMCB, MCMCgamma, MCMCint, MCMCdelta, MCMCcont, MCMCf])

labels2 = ["M", "a", "B", "gamma", "sigmaInt", "delta", "sigmaCont", "f"]

fig = corner.corner(MCMC.T, labels = labels2)

In [ ]:
lo_vals = np.zeros(4)
hi_vals = np.zeros(4)
med_vals = np.zeros(4)

plt.figure(figsize = (30, 4))
for i in range(4):
    plt.subplot(1, 6, i + 1)
    plt.title(f'Histogram of: {labels2[i + 1]}')
    plt.hist(MCMC[i], bins = 60)
    med = np.median(MCMC[i])
    plt.vlines(med, ymin = 0, ymax = 1100, linestyle = '--', color = 'black', label = f'Median: {med:.2f}')

    hi, lo = np.percentile(MCMC[i], [84, 16])

    lo_vals[i] = lo
    hi_vals[i] = hi
    med_vals[i] = med
    
    plt.axvspan(hi, lo, color = 'green', alpha = .15, label = f'Middle 68%: {lo:.2f} to {hi:.2f}')
    
    plt.ylim([0, 200])
    plt.grid(linestyle = '--', linewidth = 0.5)
    plt.legend()

Gamma and M are very strongly correlated. This makes sense as gamma only affects massive galaxies (more than 10^10 solar masses). This means that as galaxies become less massive (and dimmer) gamma will disappear from the equation making it appear as if it wasn't needed. 

In [ ]:
MAP = result.x
MAP_M = result.x[0]

In [ ]:
def M_log_liklihood(a, B, gamma, sigmaInt, delta, sigmaCont, f):

    VTCV = (df6['mB_err']**2) + (a**2 * df6['x1_err']**2) + (B**2 * df6['c_err']**2) + (2 * a * df6['cov_mB_x1']) - (2 * B * df6['cov_mB_c']) - (2 * a * B * df6['cov_x1_c'])
    
    c_light = 299792.458 #km/s
    sigmaMuPec = (5 / np.log(10)) * (250) / (c_light * df6['z'])

    sigmaSqr = VTCV + sigmaMuPec**2 + sigmaInt**2

    theta = (df6['host_logmass'] > 10).astype(int)

    r = df6['mB'] - (MAP_M + mu - a * df6['x1'] + B * df6['c'] + gamma * theta)

    L_SN = (1 / np.sqrt(2 * np.pi * sigmaSqr)) * np.exp(-.5 * r**2 / sigmaSqr) 
           
    L_cont = ( (1 / np.sqrt(2 * np.pi *sigmaCont**2) * np.exp(-0.5 * (r - delta)**2 / sigmaCont**2)) ) 

    Ltot = (1 - f) * L_SN + f * L_cont
    
    return np.sum(np.log(Ltot))

def M_neg_log_posterior(theta):

    def M_log_prior(theta):
        a, B, gamma, log_int, delta, log_cont, log_f = theta

        if 0 >= a:
            return -np.inf

        if 0 >= B:
            return -np.inf

        if ((gamma < -.2) | (.2 < gamma)):
            return -np.inf

        if (log_int < np.log(1e-4)) | (0 < log_int):
            return -np.inf

        if (delta < 0) | (1.5 < delta):
            return -np.inf

        if (log_cont < np.log(1e-4)) | (np.log(5) < log_cont):
            return -np.inf

        if (log_f < np.log(1/250)) | (np.log(.1) < log_f):
            return -np.inf

        return 0.0

    lp = M_log_prior(theta)

    if not np.isfinite(lp):
        return np.inf

    a, B, gamma, log_int, delta, log_cont, log_f = theta

    sigmaInt = np.exp(log_int)
    sigmaCont = np.exp(log_cont)
    f = np.exp(log_f)

    return -(lp + M_log_liklihood(a, B, gamma, sigmaInt, delta, sigmaCont, f))

guess = [0.15, 3.0, -.05, np.log(0.1), 1.1, np.log(1.0), np.log(.02)]
result = minimize(M_neg_log_posterior, guess)
print(result.x)


In [ ]:
def M_log_posterior(val):
    return -1 * M_neg_log_posterior(val)

pos5 = result.x + 3e-2 * np.random.randn(32, 7)

nwalkers5, ndim5 = pos5.shape

sampler5 = emcee.EnsembleSampler(nwalkers5, ndim5, M_log_posterior)

sampler5.run_mcmc(pos5, 10_000, progress = False)
print('')

In [ ]:
fig, axes = plt.subplots(7, figsize=(10, 7), sharex=True)
samples5 = sampler5.get_chain()
labels = ["a", "B", "gamma", "log_int", "delta", "log_cont", "log_f"]
for i in range(ndim5):
    ax = axes[i]
    ax.plot(samples5[:, :, i], "k", alpha=0.3)
    ax.set_xlim(0, len(samples5))
    ax.set_ylabel(labels[i])
    ax.yaxis.set_label_coords(-0.1, 0.5)

axes[-1].set_xlabel("step number")

print(f"Mean acceptance fraction: {sampler.acceptance_fraction.mean():.2f} (0.2-0.5 is healthy)")

In [ ]:
flat_samples2 = sampler5.get_chain(discard = tau_max * 3, thin = 140, flat = True)

MCMCgamma = flat_samples2.T[2]

plt.title(f'Histogram of: {labels2[3]}')
plt.hist(MCMCgamma, bins = 60)
med = np.median(MCMCgamma)
plt.vlines(med, ymin = 0, ymax = 1100, linestyle = '--', color = 'black', label = f'Median: {med:.2f}')

hi, lo = np.percentile(MCMCgamma, [84, 16])
    
plt.axvspan(hi, lo, color = 'green', alpha = .15, label = f'Middle 68%: {lo:.2f} to {hi:.2f}')
    
plt.ylim([0, 200])
plt.grid(linestyle = '--', linewidth = 0.5)
plt.legend()

The Marginalized and fixed gammas have very different uncertainties. The fixed gamma is centered about zero while the marginalized sigma is centered about ~0. The error bounds also shrink as expected between the marginal and fixed uncertainties. 

### 2d - Model comparison (10 pts)

Compare $M_0$ ($\gamma = 0$) with $M_1$ ($\gamma$ free).

1. Compute $2\ln B_{1,0}$ with the Laplace approximation (Day 13), with a prior on $\gamma$ you state and defend. How does it change if that prior is twice as wide?
2. Compute $\textrm{BIC}_0 - \textrm{BIC}_1$ for comparison.
3. How strong is your answer on Day 8's scale (0-2 weak, 2-6 positive, 6-10 strong, above 10 very strong)? Compare $\gamma$ with the published $-0.06$ mag. Could your 250 supernovae have detected a step that size? (One way: your $\sigma_\gamma$ tells you how many $\sigma$ a $-0.06$ step would be.)

In [ ]:
# 2d: your work here
def gamma_log_liklihood(M, a, B, sigmaInt, delta, sigmaCont, f):

    VTCV = (df6['mB_err']**2) + (a**2 * df6['x1_err']**2) + (B**2 * df6['c_err']**2) + (2 * a * df6['cov_mB_x1']) - (2 * B * df6['cov_mB_c']) - (2 * a * B * df6['cov_x1_c'])
    
    c_light = 299792.458 #km/s
    sigmaMuPec = (5 / np.log(10)) * (250) / (c_light * df6['z'])

    sigmaSqr = VTCV + sigmaMuPec**2 + sigmaInt**2

    r = df6['mB'] - (M + mu - a * df6['x1'] + B * df6['c'])

    L_SN = (1 / np.sqrt(2 * np.pi * sigmaSqr)) * np.exp(-.5 * r**2 / sigmaSqr) 
           
    L_cont = ( (1 / np.sqrt(2 * np.pi *sigmaCont**2) * np.exp(-0.5 * (r - delta)**2 / sigmaCont**2)) ) 

    Ltot = (1 - f) * L_SN + f * L_cont
    
    return np.sum(np.log(Ltot))

def gamma_neg_log_posterior(theta):

    def gamma_log_prior(theta):
        M, a, B, log_int, delta, log_cont, log_f = theta

        if (M < -21) | (-15 < M):
            return -np.inf
        
        if 0 >= a:
            return -np.inf

        if 0 >= B:
            return -np.inf

        if (log_int < np.log(1e-4)) | (0 < log_int):
            return -np.inf

        if (delta < 0) | (1.5 < delta):
            return -np.inf

        if (log_cont < np.log(1e-4)) | (np.log(5) < log_cont):
            return -np.inf

        if (log_f < np.log(1/250)) | (np.log(.1) < log_f):
            return -np.inf

        return 0.0

    lp = gamma_log_prior(theta)

    if not np.isfinite(lp):
        return np.inf

    M, a, B, log_int, delta, log_cont, log_f = theta

    sigmaInt = np.exp(log_int)
    sigmaCont = np.exp(log_cont)
    f = np.exp(log_f)

    return -(lp + gamma_log_liklihood(M, a, B, sigmaInt, delta, sigmaCont, f))

guess = [-19.3, 0.15, 3.0, np.log(0.1), 1.1, np.log(1.0), np.log(.02)]
result = minimize(gamma_neg_log_posterior, guess)
print(result.x)

In [ ]:
M0, a0, B0, log_sigmaInt0, delta0, log_sigmaCont0, log_f0 = result.x
LNM0 = gamma_log_liklihood(M0, a0, B0, np.exp(log_sigmaInt0), delta0, np.exp(log_sigmaCont0), np.exp(log_f0))

med_MCMCM = np.median(flat_samples.T[0])
med_MCMCa = np.median(flat_samples.T[1])
med_MCMCB = np.median(flat_samples.T[2])
med_MCMCgamma = np.median(flat_samples.T[3])
med_MCMCTlog_int = np.median(flat_samples.T[4])
med_MCMCdelta = np.median(flat_samples.T[5])
med_MCMClog_cont = np.median(flat_samples.T[6])
med_MCMClog_f = np.median(flat_samples.T[7])

LNM1 = log_liklihood(med_MCMCM, med_MCMCa, med_MCMCB, med_MCMCgamma, np.exp(med_MCMCTlog_int), med_MCMCdelta, np.exp(med_MCMClog_cont), np.exp(med_MCMClog_f))

print(LNM0)
print(LNM1)

My prior of -.2 < gamma < .2 has a width of .4. Doubling this value will make it even wider. This larger parameter space allows for more complexiity. This will cause M1 to become even more negative than it already is as it can take on even more values.

In [ ]:
B10 = 2 * (LNM1 - LNM0)
B10

In [ ]:
Baycomp = (7 * np.log(250) - 2 * LNM0) - (8 * np.log(250) - 2 * LNM1)
Baycomp

As this is ~ -7 there is strong evidence that gamma can be treated as zero. 

In [ ]:
sigma_gamma = (hi_vals[2] - lo_vals[2])/2

detection_significance = .06 / sigma_gamma

detection_significance

This would yield a .56sigma detection. This makes it very unlikely that gamma is required for this analysis.

**ANSWER:**

*M0 is the simpler model. Gamma is not required for this analysis.*

### 2e - Testing your pipeline (8 pts)

Design one test of your pipeline that would come out differently if your pipeline were wrong, and run it. State **before** you run it what result would count as a failure. One suggestion: generate fake supernovae at your redshifts, stretches and colors from parameters you choose (including a $\gamma$ you choose), fit 30 or more such datasets (maximum-likelihood fits with curvature error bars are fine here; you don't need MCMC for each), and check how often your 68% intervals contain the truth.

For my test, I will create a fake dataset using with gamma = -.06 and test to see if it recovers the value. A failure will look appear as -.06 landing outside the 1sigma range of the calucalted gamma value. This is as if the value is not within the 1sigma range, it meaens that my simulation is calaculating the values improperly and that my simulation is biased. 

In [ ]:
# 2e: your work here
#Gamma test:

theta = (df6['host_logmass'] > 10).astype(int)

gamma_s = []
for i in range(200):
    M_t = -19.5
    a_t = .3
    B_t = 3.3
    gamma_t = -.06
    sigma_int_t = .15
    delta = 1.3
    sigma_cont = 1.1
    f = .03

    mod = M_t + mu - a_t * df6['x1'] + B_t * df6['c'] + gamma_t * theta + np.random.normal(0, sigma_int_t, 250)

    df_fake = df6.copy()
    df_fake['fake'] = mod

    def fake_log_liklihood(M, a, B, gamma, sigmaInt, delta, sigmaCont, f):

        VTCV = (df6['mB_err']**2) + (a**2 * df6['x1_err']**2) + (B**2 * df6['c_err']**2) + (2 * a * df6['cov_mB_x1']) - (2 * B * df6['cov_mB_c']) - (2 * a * B * df6['cov_x1_c'])
        
        c_light = 299792.458 #km/s
        sigmaMuPec = (5 / np.log(10)) * (250) / (c_light * df6['z'])
    
        sigmaSqr = VTCV + sigmaMuPec**2 + sigmaInt**2
    
        theta = (df6['host_logmass'] > 10).astype(int)
    
        r = df_fake['fake'] - (M + mu - a * df6['x1'] + B * df6['c'] + gamma * theta)
    
        L_SN = (1 / np.sqrt(2 * np.pi * sigmaSqr)) * np.exp(-.5 * r**2 / sigmaSqr) 
               
        L_cont = ( (1 / np.sqrt(2 * np.pi *sigmaCont**2) * np.exp(-0.5 * (r - delta)**2 / sigmaCont**2)) ) 
    
        Ltot = (1 - f) * L_SN + f * L_cont
        
        return np.sum(np.log(Ltot))

    def fake_log_prior(theta):
        M, a, B, gamma, log_int, delta, log_cont, log_f = theta
        
        if (M < -21) | (-15 < M):
            return -np.inf
        
        if 0 >= a:
            return -np.inf
        
        if 0 >= B:
            return -np.inf
        
        if ((gamma < -.2) | (.2 < gamma)):
            return -np.inf
        
        if (log_int < np.log(1e-4)) | (0 < log_int):
            return -np.inf
        
        if (delta < 0) | (1.5 < delta):
            return -np.inf
        
        if (log_cont < np.log(1e-4)) | (np.log(5) < log_cont):
            return -np.inf
        
        if (log_f < np.log(1/250)) | (np.log(.1) < log_f):
            return -np.inf
        
        return 0.0

    def fake_neg_log_posterior(theta):

        lp = fake_log_prior(theta)
    
        if not np.isfinite(lp):
            return np.inf
    
        M, a, B, gamma, log_int, delta, log_cont, log_f = theta
    
        sigmaInt = np.exp(log_int)
        sigmaCont = np.exp(log_cont)
        f = np.exp(log_f)
    
        return -(lp + fake_log_liklihood(M, a, B, gamma, sigmaInt, delta, sigmaCont, f))
    

    guess = [-19.3, 0.29, 3.0, -0.05, np.log(0.15), 1.23, np.log(1.0), np.log(.02)]
    
    result = minimize(fake_neg_log_posterior, guess)
    gamma_s.append(result.x[3])

In [ ]:
plt.hist(gamma_s, bins = 10)
plt.grid(linestyle = '--', linewidth = 0.5)

gamma_s_med = np.median(gamma_s)
lo, hi = np.percentile(gamma_s, [16, 84])

plt.axvline(gamma_s_med, color = 'black', linestyle = '--', label = f'median: {gamma_s_med:.2f}')
plt.axvspan(lo, hi, color = 'green', alpha = .3)
plt.title('Coverage Test')
plt.xlabel('Gamma')
plt.ylabel('Count')

plt.legend()

My pipline recovers a gamma value of -.06 within 1sigma of the time. 

**ANSWER:**

*your answer goes here*

## Part 3: The referee report (25 pts)

White dwarfs are the cores of dead stars like the Sun. Your file `p3_wd_masses.csv` holds 600 real DA (hydrogen-atmosphere) white dwarfs from the 100 pc sample of [Kilic et al. (2020)](https://arxiv.org/abs/2006.00323), ApJ 898, 84: `name`, `teff_K`, `mass_Msun`, `mass_err_Msun`. Most single white dwarfs have masses near $0.6\,M_\odot$; Kilic et al. report a narrow peak there plus an excess of more massive ones, around $0.7$-$0.9\,M_\odot$, which merger products alone do not explain.

The notebook `labs/midterm/ai_solution.ipynb` fits your masses with a two-population model using `emcee`. Run it from that directory, after setting `NETID` in its first cell. It runs top to bottom and the plots look nice, but it's wrong in at least three substantive ways - and if you don't look at your data, you'll fool yourself the same way.

### 3a - The report (20 pts)

1. **Identify** at least three substantive errors (not style; "it used 32 walkers and I'd use 48" is not one).
2. **Demonstrate** each on *your* file: show, with a number or a plot, what the error does to the answer. A demonstration can come out negative: if an error doesn't move your numbers, show that, and say what it would take for it to matter.
3. **State** the correct treatment, and run it where you can.

In [ ]:
# 3a: your work here
df7 = load('p3_wd_masses')
df7

In [ ]:
#Ai performs incorrect sigmaclip

ai_sim_data = df7['mass_Msun'] #np.random.normal(loc = 5, size = 1000)
keep = np.abs(ai_sim_data - np.median(ai_sim_data)) < 3 * np.std(ai_sim_data)

print(3 * np.std(ai_sim_data)) 

plt.hist(ai_sim_data, bins = 60, label = 'Initial Data')
plt.hist(ai_sim_data[keep], bins = 60, label = 'Preserved Data')
plt.title('Simulated random data')
plt.xlabel('Mass (solar)')
plt.ylabel('Count')

In [ ]:
print(f'len is: {len(ai_sim_data[keep])}')

keptvals = ai_sim_data[keep]
keep2 = np.abs(keptvals - np.median(keptvals)) < 3 * np.std(keptvals)

print(f'len is: {len(keptvals[keep2])}')

The Ai's three sigma clip doesn't converge. There are still more points left to be removed iterativly. By not removing them, there are potential new outlier that sit outside the three sigma range directly affecting the population statistics. If the process doesn't converge, these valeus will remain affecting your results.

In [ ]:
#The Ai neglected errors as "they were smaller than gaussian widths":

In [ ]:
m = df7['mass_Msun']

def log_prob(p, m):
    mu1, s1, mu2, s2, f = p
    if not (0.3 < mu1 < 1.3 and 0.3 < mu2 < 1.3 and 0.005 < s1 < 0.5 and 0.005 < s2 < 0.5 and 0 < f < 1):
        return -np.inf
    return np.sum(np.log(f * stats.norm.pdf(m, mu1, s1) + (1 - f) * stats.norm.pdf(m, mu2, s2)))

# starting guess from a quick split of the sample at 0.7 Msun
lo, hi = m[m < 0.7], m[m >= 0.7]
p0 = np.array([lo.mean(), lo.std(), hi.mean(), hi.std(), lo.size / m.size])
print("starting guess:", np.round(p0, 3))

In [ ]:
log_prob(p0, m)

In [ ]:
#Versus

In [ ]:
def true_log_prob(p, m):
    mu1, s1, mu2, s2, f = p
    if not (0.3 < mu1 < 1.3 and 0.3 < mu2 < 1.3 and 0.005 < s1 < 0.5 and 0.005 < s2 < 0.5 and 0 < f < 1):
        return -np.inf
    return np.sum(np.log(f * stats.norm.pdf(m, mu1, (s1 + df7['mass_err_Msun']**2)) + (1 - f) * stats.norm.pdf(m, mu2, (s2 + df7['mass_err_Msun']**2))))

# starting guess from a quick split of the sample at 0.7 Msun
lo, hi = m[m < 0.7], m[m >= 0.7]
p0 = np.array([lo.mean(), lo.std(), hi.mean(), hi.std(), lo.size / m.size])
print("starting guess:", np.round(p0, 3))

In [ ]:
true_log_prob(p0, m)

While small, there is a shift in the log probability of the data. Not propogating these errors through is bad practice. As the errors go up, so will the liklihood value. If the errors had been larger as following, there would be a much larger difference seen:

In [ ]:
def true_log_prob(p, m):
    mu1, s1, mu2, s2, f = p
    if not (0.3 < mu1 < 1.3 and 0.3 < mu2 < 1.3 and 0.005 < s1 < 0.5 and 0.005 < s2 < 0.5 and 0 < f < 1):
        return -np.inf
    return np.sum(np.log(f * stats.norm.pdf(m, mu1, (s1 + 50 * df7['mass_err_Msun']**2)) + (1 - f) * stats.norm.pdf(m, mu2, (s2 + 50 * df7['mass_err_Msun']**2))))

# starting guess from a quick split of the sample at 0.7 Msun
lo, hi = m[m < 0.7], m[m >= 0.7]
p0 = np.array([lo.mean(), lo.std(), hi.mean(), hi.std(), lo.size / m.size])
print("starting guess:", np.round(p0, 3))

In [ ]:
true_log_prob(p0, m)

In [ ]:
#The Ai didn't allow the walkers to converge:
pos7 = result.x + 3e-2 * np.random.randn(28, 7)

nwalkers7, ndim7 = pos7.shape

sampler7 = emcee.EnsembleSampler(nwalkers7, ndim7, gamma_log_posterior)

sampler7.run_mcmc(pos7, 100, progress = False)
print('')

In [ ]:
fig, axes = plt.subplots(7, figsize=(10, 7), sharex=True)
samples7 = sampler7.get_chain()
labels = ["M", "a", "B", "log_int", "delta", "log_cont", "log_f"]
for i in range(ndim7):
    ax = axes[i]
    ax.plot(samples7[:, :, i], "k", alpha=0.3)
    ax.set_xlim(0, len(samples7))
    ax.set_ylabel(labels[i])
    ax.yaxis.set_label_coords(-0.1, 0.5)

axes[-1].set_xlabel("step number")

print(f"Mean acceptance fraction: {sampler.acceptance_fraction.mean():.2f} (0.2-0.5 is healthy)")

As the Ai's values haven't converged, the MCMC is in the process of searching the whole parameter space. This means the valeu that it comes to at the end is likely not the true value. It also doesn't result in a unified distribution across all runs. This means that the variation between each walker is very high. The Ai also hallucinates that all R-hat values are below 1.1; in reality, most are larger with some over 4.

**REFEREE REPORT:**

1) The Ai's sigma clipped once instead of clipping to convergence.
2) The Ai neglected the errors allowing for error propogation.
3) The Ai didn't allow its sampler to converge. 

### 3b - AI-use appendix (5 pts)

1. Which tools did you use, for what, what did they get wrong, and how did you catch it? "I used ChatGPT to help" is useless; "none" is fine if that's true. I promise I don't care which - just be honest about what you did.
2. What would have gone wrong in your answer if you had not checked the AI's output? Name the specific thing, in two or three sentences.

**AI-USE APPENDIX:**

*I used Copilot to assit in this analysis. It got my liklihood functions mixed up. Since I made multiple definitions it thought I was talking about a different function than the one I wanted an update on. The Ai then fed me the wrong suggestions for what to change in the liklihood function (it suggested that I remove some of my priors). If I had not checked which function I was working on then I would've removed my gamma priors from my Magnitude log liklihood function. This would've removed cruicial informaiton that would help the pipline converge to the proper answer.*

Get started. You know how to do all of this. If not, there's something in a lab or class notebook that can help. Slack me with questions. Best of luck.